# 03 Tabular and CNN Evaluation at 500 m

This notebook is the final controlled multimodal experiment after the 500 m feature audit and XGBoost model selection.

It contains four logically separate analyses:

**A. Final multimodal ablation**  
- `Tabular` — the final 36 engineered predictors.  
- `Tabular + CNN` — the same 36 predictors plus frozen **ImageNet ResNet50** visual embeddings.

**B. Raw vs log1p target sensitivity**  
- uses **Tabular XGBoost only**;  
- `log1p` predictions are back-transformed with `expm1()` before R², RMSE and MAE are evaluated on the original electricity-consumption scale.

**C. PCA sensitivity / tuning**  
- PCA is never selected on the full dataset;  
- the explained-variance threshold is tuned **inside each nested-CV training fold** from 70%, 75%, 80%, 85% and 90%;  
- the outer test fold is transformed only after PCA has been fitted within the training procedure.

**D. Supplementary pretraining experiment**  
- architecture is fixed to **ResNet50**;  
- compares ImageNet, Satlas and SSL4EO-S12 visual representations under the same nested-CV XGBoost framework;  
- this is supplementary sensitivity evidence and does not alter the predefined main ImageNet experiment automatically.

Validation is held fixed throughout:
- Random 5-fold CV;
- KMeans K=4 spatial leave-one-block-out CV using the assignments produced upstream;
- nested XGBoost hyperparameter tuning;
- fold-wise median imputation;
- fold-wise PCA for CNN embeddings;
- identical outer folds and XGBoost search spaces where configurations are compared.

The four legacy output files required by notebook 004 are written to `work/multimodal_fusion_500m` with their original filenames.

In [ ]:
# ============================================================
# Imports and global display settings
# ============================================================

import gc
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)
from sklearn.model_selection import (
    KFold,
    GroupKFold,
    RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
SET2_PALETTE = sns.color_palette("Set2", n_colors=8)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# ============================================================
# Paths, final features and experiment settings
# ============================================================

import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from project_config import RAW_DATA_DIR, WORK_DIR

PROJECT_DATA_DIR = WORK_DIR

FEATURE_DIR = (
    PROJECT_DATA_DIR
    / "grid_size_selection"
    / "features"
)

GRID_DIR = (
    PROJECT_DATA_DIR
    / "grid_size_selection"
    / "grids"
)

FEATURE_MATRIX_PATH = (
    FEATURE_DIR
    / "feature_matrix_500m.csv"
)

GRID_PATH = (
    GRID_DIR
    / "grid_features_500m.gpkg"
)

SPATIAL_BLOCK_PATH = (
    PROJECT_DATA_DIR
    / "model_tuning_500m"
    / "spatial_block_assignments_500m.csv"
)

# Main output path is intentionally kept compatible with notebook 004.
OUTPUT_DIR = (
    PROJECT_DATA_DIR
    / "multimodal_fusion_500m"
)

FIGURE_DIR = OUTPUT_DIR / "figures"
SUPPLEMENTARY_DIR = OUTPUT_DIR / "supplementary_pretraining"
SUPPLEMENTARY_FIGURE_DIR = SUPPLEMENTARY_DIR / "figures"

for directory in [
    OUTPUT_DIR,
    FIGURE_DIR,
    SUPPLEMENTARY_DIR,
    SUPPLEMENTARY_FIGURE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

TARGET_COLUMN = "elec_consumption"

BASELINE_FEATURES = [
    "area_m2",
    "B02_mean", "B02_std", "B02_max",
    "B03_mean", "B03_std", "B03_max",
    "B04_mean", "B04_std", "B04_max",
    "B08_mean", "B08_std", "B08_max",
    "B11_mean", "B11_std", "B11_max",
    "NDVI", "NDBI",
    "NTL_mean", "NTL_max",
    "dist_major_road",
    "road_count",
    "road_length_major",
    "road_density",
    "major_road_ratio",
    "poi_economic_count",
    "poi_social_count",
    "poi_other_count",
    "poi_shannon",
    "dist_economic_poi",
    "dist_social_poi",
    "pop_density_km2",
    "building_count",
    "building_area_mean",
    "building_area_std",
    "building_coverage",
]

assert len(BASELINE_FEATURES) == 36

# New 001 currently saves ImageNet ResNet50 under this name.
IMAGENET_R50_CANDIDATE_PATHS = [
    FEATURE_DIR / "cnn_features_500m_resnet50_raw.csv",
    FEATURE_DIR / "cnn_features_500m_resnet50_imagenet_raw.csv",
]

SATLAS_R50_PATH = (
    FEATURE_DIR
    / "cnn_features_500m_resnet50_satlas_raw.csv"
)

SSL4EO_R50_PATH = (
    FEATURE_DIR
    / "cnn_features_500m_resnet50_ssl4eo_s12_raw.csv"
)

RANDOM_FOLDS = 5
INNER_FOLDS = 3
SPATIAL_CLUSTERS = 4
SEARCH_ITERATIONS = 30

PCA_VARIANCE_OPTIONS = [
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
]

DEFAULT_PCA_VARIANCE = 0.80
FIGURE_DPI = 300

# Supplementary experiment switches.
RUN_SUPPLEMENTARY_PRETRAINING = True
EXTRACT_MISSING_PRETRAINING_EMBEDDINGS = True
SUPPLEMENTARY_BATCH_SIZE = 64
SUPPLEMENTARY_PATCH_SIZE = 256

VALIDATION_TITLES = {
    "random_5fold": "Random 5-fold CV",
    "spatial_kmeans_leave_one_out": "Spatial KMeans K=4 CV",
}

for path in [
    FEATURE_MATRIX_PATH,
    GRID_PATH,
    SPATIAL_BLOCK_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required upstream file not found: {path}"
        )

print("Final tabular predictors:", len(BASELINE_FEATURES))
print("PCA variance candidates:", PCA_VARIANCE_OPTIONS)
print("XGBoost search iterations:", SEARCH_ITERATIONS)
print("Main output directory:", OUTPUT_DIR)

In [ ]:
# ============================================================
# Shared file and embedding helpers
# ============================================================

def resolve_existing_path(candidate_paths, label):
    for path in candidate_paths:
        if path.exists():
            return path

    raise FileNotFoundError(
        f"{label} was not found. Checked:\n"
        + "\n".join(str(path) for path in candidate_paths)
    )


def standardise_embedding_table(
    table,
    expected_dimension=2048,
    source_label="CNN",
):
    if "grid_id" not in table.columns:
        raise KeyError(
            f"{source_label} embedding table must contain 'grid_id'."
        )

    if table["grid_id"].duplicated().any():
        raise ValueError(
            f"Duplicate grid_id values found in {source_label} embeddings."
        )

    raw_feature_columns = [
        column
        for column in table.columns
        if column != "grid_id"
    ]

    if len(raw_feature_columns) != expected_dimension:
        raise ValueError(
            f"{source_label}: expected {expected_dimension} embedding dimensions, "
            f"found {len(raw_feature_columns)}."
        )

    feature_values = table[
        raw_feature_columns
    ].to_numpy(dtype=np.float32)

    if not np.isfinite(feature_values).all():
        raise ValueError(
            f"Non-finite values found in {source_label} embeddings."
        )

    standard_columns = [
        f"cnn_{index}"
        for index in range(expected_dimension)
    ]

    standardised = pd.DataFrame(
        feature_values,
        columns=standard_columns,
    )
    standardised.insert(
        0,
        "grid_id",
        table["grid_id"].to_numpy(),
    )

    return (
        standardised
        .sort_values("grid_id")
        .reset_index(drop=True)
    )


def load_embedding_table(
    path,
    source_label,
    expected_dimension=2048,
):
    table = pd.read_csv(path)
    table = standardise_embedding_table(
        table,
        expected_dimension=expected_dimension,
        source_label=source_label,
    )

    print(
        f"{source_label}: {len(table)} samples × "
        f"{expected_dimension} embedding dimensions"
    )
    print("Source file:", path)

    return table

In [ ]:
# ============================================================
# Reproducibility check for final Tabular experiment
# ============================================================

print("Samples:", len(main_df))
print("Tabular features:", len(BASELINE_FEATURES))
print("Search iterations:", SEARCH_ITERATIONS)
print("Random state:", RANDOM_STATE)

print("\nFirst 5 grid IDs:")
print(main_df["grid_id"].head().tolist())

print("\nLast 5 grid IDs:")
print(main_df["grid_id"].tail().tolist())

print("\nSpatial block sizes:")
print(
    main_df["spatial_block"]
    .value_counts()
    .sort_index()
)

print("\nRandom outer-fold test sizes:")
for split in random_outer_splits:
    print(
        split["fold"],
        len(split["test_index"])
    )

# A. Final multimodal ablation

The main experiment is deliberately limited to the final model question:

- **Tabular** → 36 final predictors → XGBoost.
- **Tabular + CNN** → the same 36 predictors + frozen ImageNet ResNet50 embeddings → nested fold-wise PCA → XGBoost.

`feature_set` is kept as `Tabular` / `Tabular + CNN` in the exported tables so notebook 004 remains compatible. The explicit CNN source is stored separately as `cnn_source = ImageNet ResNet50`.

In [ ]:
# ============================================================
# Load and align tabular data, spatial folds and ImageNet ResNet50
# ============================================================

IMAGENET_R50_PATH = resolve_existing_path(
    IMAGENET_R50_CANDIDATE_PATHS,
    "ImageNet ResNet50 raw embeddings",
)

imagenet_df = load_embedding_table(
    IMAGENET_R50_PATH,
    source_label="ImageNet ResNet50",
    expected_dimension=2048,
)

CNN_FEATURES = [
    f"cnn_{index}"
    for index in range(2048)
]

tabular_df = pd.read_csv(
    FEATURE_MATRIX_PATH
)

spatial_df = pd.read_csv(
    SPATIAL_BLOCK_PATH
)

required_tabular_columns = [
    "grid_id",
    "centroid_x",
    "centroid_y",
    TARGET_COLUMN,
] + BASELINE_FEATURES

missing_tabular_columns = [
    column
    for column in required_tabular_columns
    if column not in tabular_df.columns
]

if missing_tabular_columns:
    raise KeyError(
        f"Missing final tabular columns: {missing_tabular_columns}"
    )

required_spatial_columns = [
    "grid_id",
    "spatial_block",
]

missing_spatial_columns = [
    column
    for column in required_spatial_columns
    if column not in spatial_df.columns
]

if missing_spatial_columns:
    raise KeyError(
        f"Missing spatial-fold columns: {missing_spatial_columns}"
    )

if tabular_df["grid_id"].duplicated().any():
    raise ValueError("Duplicate grid_id values found in tabular data.")

if spatial_df["grid_id"].duplicated().any():
    raise ValueError("Duplicate grid_id values found in spatial-fold data.")

BASE_FRAME = (
    tabular_df[
        required_tabular_columns
    ]
    .merge(
        spatial_df[
            required_spatial_columns
        ],
        on="grid_id",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("grid_id")
    .reset_index(drop=True)
)

if set(BASE_FRAME["grid_id"]) != set(imagenet_df["grid_id"]):
    raise ValueError(
        "ImageNet ResNet50 grid IDs do not match the final 500 m modelling matrix."
    )

main_df = (
    BASE_FRAME
    .merge(
        imagenet_df,
        on="grid_id",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("grid_id")
    .reset_index(drop=True)
)

if main_df[TARGET_COLUMN].isna().any():
    raise ValueError("Target contains missing values.")

print("Modelling samples:", len(main_df))
print("Tabular predictors:", len(BASELINE_FEATURES))
print("ImageNet ResNet50 dimensions:", len(CNN_FEATURES))
print(
    "Spatial blocks:",
    main_df["spatial_block"]
    .value_counts()
    .sort_index()
    .to_dict(),
)

display(
    main_df[
        ["grid_id", TARGET_COLUMN, "spatial_block"]
    ].head()
)

In [ ]:
# ============================================================
# Define fixed outer validation splits
# ============================================================

random_cv = KFold(
    n_splits=RANDOM_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

random_outer_splits = []

for fold, (train_idx, test_idx) in enumerate(
    random_cv.split(main_df),
    start=1,
):
    random_outer_splits.append({
        "fold": fold,
        "train_index": train_idx,
        "test_index": test_idx,
    })

spatial_blocks_found = sorted(
    main_df["spatial_block"].unique()
)

if len(spatial_blocks_found) != SPATIAL_CLUSTERS:
    raise ValueError(
        f"Expected {SPATIAL_CLUSTERS} spatial blocks, "
        f"found {len(spatial_blocks_found)}: {spatial_blocks_found}"
    )

spatial_outer_splits = []

for fold, block in enumerate(
    spatial_blocks_found,
    start=1,
):
    test_idx = np.where(
        main_df["spatial_block"].to_numpy() == block
    )[0]

    train_idx = np.where(
        main_df["spatial_block"].to_numpy() != block
    )[0]

    spatial_outer_splits.append({
        "fold": fold,
        "spatial_block": int(block),
        "train_index": train_idx,
        "test_index": test_idx,
    })

VALIDATION_SCHEMES = {
    "random_5fold": random_outer_splits,
    "spatial_kmeans_leave_one_out": spatial_outer_splits,
}

for validation_name, splits in VALIDATION_SCHEMES.items():
    print("\n" + VALIDATION_TITLES[validation_name])
    for split in splits:
        print(
            f"Fold {split['fold']}: "
            f"train={len(split['train_index'])}, "
            f"test={len(split['test_index'])}"
        )

In [ ]:
# ============================================================
# XGBoost model and shared search space
# ============================================================

def make_xgb_model():
    return XGBRegressor(
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbosity=0,
    )


XGB_SEARCH_SPACE = {
    "model__n_estimators": [
        300, 500, 800, 1200
    ],
    "model__learning_rate": [
        0.01, 0.03, 0.05, 0.08
    ],
    "model__max_depth": [
        2, 3, 4, 5, 6
    ],
    "model__min_child_weight": [
        1, 3, 5, 10
    ],
    "model__subsample": [
        0.60, 0.75, 0.90, 1.0
    ],
    "model__colsample_bytree": [
        0.50, 0.70, 0.90, 1.0
    ],
    "model__reg_alpha": [
        0, 0.01, 0.1, 0.5, 1
    ],
    "model__reg_lambda": [
        1, 3, 5, 10, 20
    ],
    "model__gamma": [
        0, 0.05, 0.10, 0.25
    ],
}

print("PCA is tuned only for CNN configurations.")
print("PCA candidates:", PCA_VARIANCE_OPTIONS)

In [ ]:
# ============================================================
# Leakage-safe preprocessing pipeline
# ============================================================

def make_pipeline(
    use_cnn=False,
    cnn_features=None,
):
    transformers = [
        (
            "tabular",
            SimpleImputer(strategy="median"),
            BASELINE_FEATURES,
        )
    ]

    if use_cnn:
        if cnn_features is None:
            raise ValueError(
                "cnn_features must be provided when use_cnn=True."
            )

        cnn_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "pca",
                PCA(
                    n_components=DEFAULT_PCA_VARIANCE,
                    svd_solver="full",
                ),
            ),
        ])

        transformers.append(
            (
                "cnn",
                cnn_pipeline,
                cnn_features,
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "model",
            make_xgb_model(),
        ),
    ])

In [ ]:
# ============================================================
# Shared metrics and inner validation
# ============================================================

def regression_metrics(
    y_true,
    y_pred,
):
    return {
        "R2": r2_score(
            y_true,
            y_pred,
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                y_true,
                y_pred,
            )
        ),
        "MAE": mean_absolute_error(
            y_true,
            y_pred,
        ),
    }


def make_inner_cv(
    train_frame,
    validation_name,
    fold_seed,
):
    if validation_name == "random_5fold":
        inner_cv = KFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=fold_seed,
        )
        return inner_cv, None

    coordinates = train_frame[
        ["centroid_x", "centroid_y"]
    ].to_numpy()

    inner_kmeans = KMeans(
        n_clusters=INNER_FOLDS,
        random_state=fold_seed,
        n_init=10,
    )

    groups = inner_kmeans.fit_predict(
        coordinates
    )

    inner_cv = GroupKFold(
        n_splits=INNER_FOLDS
    )

    return inner_cv, groups

In [ ]:
# ============================================================
# General nested-CV experiment runner
# ============================================================

def run_nested_experiment(
    frame,
    splits,
    validation_name,
    feature_set,
    cnn_source="None",
    cnn_features=None,
    target_mode="raw",
):
    use_cnn = cnn_features is not None

    input_columns = BASELINE_FEATURES.copy()
    if use_cnn:
        input_columns += list(cnn_features)

    y_raw_all = frame[
        TARGET_COLUMN
    ].to_numpy(dtype=float)

    fold_rows = []
    prediction_frames = []

    for split in splits:
        fold = split["fold"]
        train_idx = split["train_index"]
        test_idx = split["test_index"]

        train_frame = frame.iloc[
            train_idx
        ].copy()

        test_frame = frame.iloc[
            test_idx
        ].copy()

        X_train = train_frame[
            input_columns
        ]

        X_test = test_frame[
            input_columns
        ]

        y_train_raw = y_raw_all[
            train_idx
        ]

        y_test_raw = y_raw_all[
            test_idx
        ]

        if target_mode == "raw":
            y_train_fit = y_train_raw
        elif target_mode == "log1p":
            y_train_fit = np.log1p(
                y_train_raw
            )
        else:
            raise ValueError(
                f"Unknown target_mode: {target_mode}"
            )

        fold_seed = RANDOM_STATE + fold

        inner_cv, inner_groups = make_inner_cv(
            train_frame,
            validation_name,
            fold_seed,
        )

        pipeline = make_pipeline(
            use_cnn=use_cnn,
            cnn_features=cnn_features,
        )

        search_space = XGB_SEARCH_SPACE.copy()

        if use_cnn:
            search_space[
                "preprocessor__cnn__pca__n_components"
            ] = PCA_VARIANCE_OPTIONS

        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=search_space,
            n_iter=SEARCH_ITERATIONS,
            scoring="r2",
            cv=inner_cv,
            random_state=fold_seed,
            n_jobs=-1,
            refit=True,
            verbose=0,
        )

        fit_kwargs = {}
        if inner_groups is not None:
            fit_kwargs["groups"] = inner_groups

        search.fit(
            X_train,
            y_train_fit,
            **fit_kwargs,
        )

        best_model = search.best_estimator_

        train_pred_fit = best_model.predict(
            X_train
        )

        test_pred_fit = best_model.predict(
            X_test
        )

        if target_mode == "log1p":
            train_pred = np.expm1(
                train_pred_fit
            ).clip(min=0)

            test_pred = np.expm1(
                test_pred_fit
            ).clip(min=0)
        else:
            train_pred = train_pred_fit
            test_pred = test_pred_fit

        test_metrics = regression_metrics(
            y_test_raw,
            test_pred,
        )

        train_r2 = r2_score(
            y_train_raw,
            train_pred,
        )

        selected_pca_variance = np.nan
        n_pca_components = np.nan

        if use_cnn:
            selected_pca_variance = (
                search.best_params_[
                    "preprocessor__cnn__pca__n_components"
                ]
            )

            pca = (
                best_model
                .named_steps["preprocessor"]
                .named_transformers_["cnn"]
                .named_steps["pca"]
            )

            n_pca_components = int(
                pca.n_components_
            )

        fold_rows.append({
            "validation": validation_name,
            "feature_set": feature_set,
            "cnn_source": cnn_source,
            "target_mode": target_mode,
            "fold": fold,
            "R2": test_metrics["R2"],
            "RMSE": test_metrics["RMSE"],
            "MAE": test_metrics["MAE"],
            "train_R2": train_r2,
            "inner_best_R2_fit_scale": search.best_score_,
            "selected_pca_variance": selected_pca_variance,
            "n_pca_components": n_pca_components,
            "best_params": json.dumps(
                search.best_params_,
                sort_keys=True,
            ),
        })

        prediction_frames.append(
            pd.DataFrame({
                "grid_id": test_frame[
                    "grid_id"
                ].to_numpy(),
                "validation": validation_name,
                "feature_set": feature_set,
                "cnn_source": cnn_source,
                "target_mode": target_mode,
                "fold": fold,
                "observed": y_test_raw,
                "predicted": test_pred,
                "residual": y_test_raw - test_pred,
            })
        )

        message = (
            f"{validation_name} | {feature_set} | {cnn_source} | "
            f"target={target_mode} | fold {fold} | "
            f"R²={test_metrics['R2']:.4f} | "
            f"train R²={train_r2:.4f} | "
            f"inner R²={search.best_score_:.4f}"
        )

        if use_cnn:
            message += (
                f" | PCA={selected_pca_variance:.2f}"
                f" | components={n_pca_components}"
            )

        print(message)

    return (
        pd.DataFrame(fold_rows),
        pd.concat(
            prediction_frames,
            ignore_index=True,
        ),
    )

In [ ]:
# ============================================================
# Shared result summariser
# ============================================================

def summarise_results(
    fold_results,
    predictions,
    group_columns,
):
    fold_summary = (
        fold_results
        .groupby(
            group_columns,
            as_index=False,
        )
        .agg(
            n_folds=("fold", "count"),
            R2_mean=("R2", "mean"),
            R2_std=("R2", "std"),
            RMSE_mean=("RMSE", "mean"),
            RMSE_std=("RMSE", "std"),
            MAE_mean=("MAE", "mean"),
            MAE_std=("MAE", "std"),
            train_R2_mean=("train_R2", "mean"),
            PCA_components_mean=(
                "n_pca_components",
                "mean",
            ),
        )
    )

    oof_rows = []

    for keys, group in predictions.groupby(
        group_columns
    ):
        if not isinstance(keys, tuple):
            keys = (keys,)

        metrics = regression_metrics(
            group["observed"].to_numpy(),
            group["predicted"].to_numpy(),
        )

        row = dict(
            zip(
                group_columns,
                keys,
            )
        )

        row.update({
            "OOF_R2": metrics["R2"],
            "OOF_RMSE": metrics["RMSE"],
            "OOF_MAE": metrics["MAE"],
        })

        oof_rows.append(row)

    oof_summary = pd.DataFrame(
        oof_rows
    )

    return fold_summary.merge(
        oof_summary,
        on=group_columns,
        how="left",
        validate="one_to_one",
    )

In [ ]:
# ============================================================
# Run A: final Tabular vs Tabular + ImageNet ResNet50 ablation
# ============================================================

main_fold_frames = []
main_prediction_frames = []

for validation_name, splits in VALIDATION_SCHEMES.items():
    print("\n" + "=" * 78)
    print(VALIDATION_TITLES[validation_name])

    tabular_folds, tabular_predictions = run_nested_experiment(
        frame=main_df,
        splits=splits,
        validation_name=validation_name,
        feature_set="Tabular",
        cnn_source="None",
        cnn_features=None,
        target_mode="raw",
    )

    fusion_folds, fusion_predictions_part = run_nested_experiment(
        frame=main_df,
        splits=splits,
        validation_name=validation_name,
        feature_set="Tabular + CNN",
        cnn_source="ImageNet ResNet50",
        cnn_features=CNN_FEATURES,
        target_mode="raw",
    )

    main_fold_frames.extend([
        tabular_folds,
        fusion_folds,
    ])

    main_prediction_frames.extend([
        tabular_predictions,
        fusion_predictions_part,
    ])

fusion_fold_results = pd.concat(
    main_fold_frames,
    ignore_index=True,
)

fusion_predictions = pd.concat(
    main_prediction_frames,
    ignore_index=True,
)

fusion_summary = summarise_results(
    fusion_fold_results,
    fusion_predictions,
    group_columns=[
        "validation",
        "feature_set",
        "cnn_source",
    ],
)

display(
    fusion_summary
    .sort_values(
        ["validation", "OOF_R2"],
        ascending=[True, False],
    )
    .round(4)
)

In [ ]:
# ============================================================
# Incremental effect of ImageNet ResNet50 visual features
# ============================================================

gain_rows = []

for validation_name in fusion_summary[
    "validation"
].unique():
    subset = fusion_summary[
        fusion_summary["validation"] == validation_name
    ].set_index("feature_set")

    tabular = subset.loc["Tabular"]
    fusion = subset.loc["Tabular + CNN"]

    gain_rows.append({
        "validation": validation_name,
        "cnn_source": "ImageNet ResNet50",
        "delta_R2_mean": (
            fusion["R2_mean"]
            - tabular["R2_mean"]
        ),
        "delta_OOF_R2": (
            fusion["OOF_R2"]
            - tabular["OOF_R2"]
        ),
        "delta_RMSE_mean": (
            fusion["RMSE_mean"]
            - tabular["RMSE_mean"]
        ),
        "delta_OOF_RMSE": (
            fusion["OOF_RMSE"]
            - tabular["OOF_RMSE"]
        ),
        "delta_MAE_mean": (
            fusion["MAE_mean"]
            - tabular["MAE_mean"]
        ),
        "delta_OOF_MAE": (
            fusion["OOF_MAE"]
            - tabular["OOF_MAE"]
        ),
    })

gain_table = pd.DataFrame(
    gain_rows
)

display(
    gain_table.round(4)
)

In [ ]:
# ============================================================
# Spatial fold-by-fold comparison for the main experiment
# ============================================================

spatial_fold_comparison = (
    fusion_fold_results[
        fusion_fold_results["validation"]
        == "spatial_kmeans_leave_one_out"
    ]
    .pivot(
        index="fold",
        columns="feature_set",
        values=[
            "R2",
            "RMSE",
            "MAE",
            "train_R2",
        ],
    )
)

display(
    spatial_fold_comparison.round(4)
)

In [ ]:
# ============================================================
# Figure A: unified main multimodal R2 comparison
# ============================================================

MAIN_DISPLAY_LABELS = {
    "Tabular": "Tabular",
    "Tabular + CNN": "Tabular + ImageNet\nResNet50",
}

plot_data = fusion_fold_results.copy()
plot_data["display_label"] = plot_data[
    "feature_set"
].map(MAIN_DISPLAY_LABELS)

main_order = [
    "Tabular",
    "Tabular + ImageNet\nResNet50",
]

main_palette = {
    main_order[0]: SET2_PALETTE[0],
    main_order[1]: SET2_PALETTE[1],
}

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11.5, 4.8),
)

for ax, validation_name in zip(
    axes,
    VALIDATION_TITLES,
):
    subset = plot_data[
        plot_data["validation"] == validation_name
    ]

    sns.barplot(
        data=subset,
        x="display_label",
        y="R2",
        order=main_order,
        errorbar="sd",
        palette=main_palette,
        ax=ax,
    )

    ax.axhline(
        0,
        color="black",
        linewidth=0.8,
    )

    ax.set_title(
        VALIDATION_TITLES[validation_name]
    )
    ax.set_xlabel("")
    ax.set_ylabel("R²")

fig.suptitle(
    "Final Multimodal Ablation — ImageNet ResNet50",
    y=1.02,
)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "A_final_multimodal_ablation_r2.png",
    dpi=FIGURE_DPI,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# ============================================================
# Save A outputs using notebook-004-compatible filenames
# ============================================================

fusion_fold_results.to_csv(
    OUTPUT_DIR / "multimodal_nested_fold_results.csv",
    index=False,
)

fusion_predictions.to_csv(
    OUTPUT_DIR / "multimodal_oof_predictions.csv",
    index=False,
)

fusion_summary.to_csv(
    OUTPUT_DIR / "multimodal_summary.csv",
    index=False,
)

gain_table.to_csv(
    OUTPUT_DIR / "cnn_incremental_gain.csv",
    index=False,
)

spatial_fold_comparison.to_csv(
    OUTPUT_DIR / "main_spatial_fold_comparison.csv"
)

print("Saved main multimodal outputs to:")
print(OUTPUT_DIR)

In [ ]:
# ============================================================
# Final Tabular XGBoost performance summary
# Random CV vs Spatial CV
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Keep final raw-target Tabular XGBoost results
# ------------------------------------------------------------

tabular_folds = (
    fusion_fold_results[
        (fusion_fold_results["feature_set"] == "Tabular")
        & (fusion_fold_results["target_mode"] == "raw")
    ]
    .copy()
)

tabular_predictions = (
    fusion_predictions[
        (fusion_predictions["feature_set"] == "Tabular")
        & (fusion_predictions["target_mode"] == "raw")
    ]
    .copy()
)


# ------------------------------------------------------------
# 2. Fold-level summary
# ------------------------------------------------------------

summary_rows = []

validation_order = [
    "random_5fold",
    "spatial_kmeans_leave_one_out",
]

validation_labels = {
    "random_5fold": "Random 5-fold CV",
    "spatial_kmeans_leave_one_out": "Spatial KMeans CV",
}


for validation in validation_order:

    # Fold-level results
    fold_df = (
        tabular_folds[
            tabular_folds["validation"] == validation
        ]
        .sort_values("fold")
        .copy()
    )

    # OOF predictions
    pred_df = (
        tabular_predictions[
            tabular_predictions["validation"] == validation
        ]
        .copy()
    )

    # ----------------------------------------
    # Fold R² values
    # ----------------------------------------

    fold_r2_values = (
        fold_df["R2"]
        .to_numpy()
    )

    fold_r2_string = ", ".join(
        f"{value:.3f}"
        for value in fold_r2_values
    )

    # ----------------------------------------
    # Fold-level statistics
    # ----------------------------------------

    r2_mean = fold_df["R2"].mean()

    r2_std = fold_df["R2"].std(
        ddof=0
    )

    train_r2_mean = (
        fold_df["train_R2"].mean()
    )

    generalisation_gap = (
        train_r2_mean - r2_mean
    )

    # ----------------------------------------
    # Pooled OOF metrics
    # ----------------------------------------

    oof_metrics = regression_metrics(
        pred_df["observed"],
        pred_df["predicted"],
    )

    # ----------------------------------------
    # Store
    # ----------------------------------------

    summary_rows.append({
        "Validation":
            validation_labels[validation],

        "N folds":
            len(fold_df),

        "N predictions":
            len(pred_df),

        "Fold R²":
            fold_r2_string,

        "Mean R²":
            r2_mean,

        "R² SD":
            r2_std,

        "Mean Train R²":
            train_r2_mean,

        "Generalisation Gap":
            generalisation_gap,

        "OOF R²":
            oof_metrics["R2"],

        "OOF RMSE":
            oof_metrics["RMSE"],

        "OOF MAE":
            oof_metrics["MAE"],
    })


# ------------------------------------------------------------
# 3. Create table
# ------------------------------------------------------------

final_tabular_summary = pd.DataFrame(
    summary_rows
)


# ------------------------------------------------------------
# 4. Round numerical columns for thesis display
# ------------------------------------------------------------

display_table = (
    final_tabular_summary
    .copy()
)

round_columns = [
    "Mean R²",
    "R² SD",
    "Mean Train R²",
    "Generalisation Gap",
    "OOF R²",
]

display_table[round_columns] = (
    display_table[round_columns]
    .round(3)
)

display_table[
    ["OOF RMSE", "OOF MAE"]
] = (
    display_table[
        ["OOF RMSE", "OOF MAE"]
    ]
    .round(2)
)


# ------------------------------------------------------------
# 5. Display
# ------------------------------------------------------------

display(
    display_table
)


# ------------------------------------------------------------
# 6. Save
# ------------------------------------------------------------

output_path = (
    OUTPUT_DIR
    / "final_tabular_xgboost_performance_summary.csv"
)

final_tabular_summary.to_csv(
    output_path,
    index=False,
)

print(
    f"Saved: {output_path}"
)

# B. Raw vs log1p target sensitivity — Tabular XGBoost only

The raw-target result is reused from section A. Only the `log1p` Tabular XGBoost models are newly fitted here. All outer-fold predictions and training predictions are back-transformed to the original electricity-consumption scale before evaluation.

In [ ]:
# ============================================================
# Run B: log1p sensitivity using Tabular XGBoost only
# ============================================================

log_fold_frames = []
log_prediction_frames = []

for validation_name, splits in VALIDATION_SCHEMES.items():
    print("\n" + "=" * 78)
    print(
        VALIDATION_TITLES[validation_name],
        "| Tabular | log1p target",
    )

    fold_result, predictions = run_nested_experiment(
        frame=main_df,
        splits=splits,
        validation_name=validation_name,
        feature_set="Tabular",
        cnn_source="None",
        cnn_features=None,
        target_mode="log1p",
    )

    log_fold_frames.append(fold_result)
    log_prediction_frames.append(predictions)

log_fold_results = pd.concat(
    log_fold_frames,
    ignore_index=True,
)

log_oof_predictions = pd.concat(
    log_prediction_frames,
    ignore_index=True,
)

log_target_summary = summarise_results(
    log_fold_results,
    log_oof_predictions,
    group_columns=[
        "validation",
        "target_mode",
    ],
)

# ============================================================
# Reuse the latest raw-target Tabular results from Section A
# ============================================================

raw_target_fold_results = pd.read_csv(
    OUTPUT_DIR / "multimodal_nested_fold_results.csv"
)

raw_target_fold_results = (
    raw_target_fold_results[
        (raw_target_fold_results["feature_set"] == "Tabular")
        & (raw_target_fold_results["target_mode"] == "raw")
    ]
    .copy()
)

raw_target_predictions = pd.read_csv(
    OUTPUT_DIR / "multimodal_oof_predictions.csv"
)

raw_target_predictions = (
    raw_target_predictions[
        (raw_target_predictions["feature_set"] == "Tabular")
        & (raw_target_predictions["target_mode"] == "raw")
    ]
    .copy()
)


raw_target_summary = summarise_results(
    raw_target_fold_results,
    raw_target_predictions,
    group_columns=[
        "validation",
        "target_mode",
    ],
)

target_comparison = pd.concat(
    [
        raw_target_summary,
        log_target_summary,
    ],
    ignore_index=True,
)

target_comparison[
    "R2_generalisation_gap"
] = (
    target_comparison["train_R2_mean"]
    - target_comparison["R2_mean"]
)

target_comparison = (
    target_comparison
    .sort_values(
        ["validation", "target_mode"]
    )
    .reset_index(drop=True)
)

display(
    target_comparison.round(4)
)

In [ ]:
# ============================================================
# Quantify log1p change relative to raw target
# ============================================================

target_delta_rows = []

for validation_name in target_comparison[
    "validation"
].unique():
    subset = target_comparison[
        target_comparison["validation"] == validation_name
    ].set_index("target_mode")

    raw = subset.loc["raw"]
    log1p = subset.loc["log1p"]

    target_delta_rows.append({
        "validation": validation_name,
        "delta_R2_mean_log_minus_raw": (
            log1p["R2_mean"]
            - raw["R2_mean"]
        ),
        "delta_OOF_R2_log_minus_raw": (
            log1p["OOF_R2"]
            - raw["OOF_R2"]
        ),
        "delta_RMSE_mean_log_minus_raw": (
            log1p["RMSE_mean"]
            - raw["RMSE_mean"]
        ),
        "delta_OOF_RMSE_log_minus_raw": (
            log1p["OOF_RMSE"]
            - raw["OOF_RMSE"]
        ),
        "delta_MAE_mean_log_minus_raw": (
            log1p["MAE_mean"]
            - raw["MAE_mean"]
        ),
        "delta_OOF_MAE_log_minus_raw": (
            log1p["OOF_MAE"]
            - raw["OOF_MAE"]
        ),
        "delta_generalisation_gap_log_minus_raw": (
            log1p["R2_generalisation_gap"]
            - raw["R2_generalisation_gap"]
        ),
    })

target_delta = pd.DataFrame(
    target_delta_rows
)

display(
    target_delta.round(4)
)

In [ ]:
# ============================================================
# Figure B: unified raw vs log1p R2 comparison
# ============================================================

raw_plot = raw_target_fold_results[
    ["validation", "fold", "R2"]
].copy()
raw_plot["target"] = "Raw"

log_plot = log_fold_results[
    ["validation", "fold", "R2"]
].copy()
log_plot["target"] = "log1p"

target_plot_data = pd.concat(
    [raw_plot, log_plot],
    ignore_index=True,
)

target_order = ["Raw", "log1p"]
target_palette = {
    "Raw": SET2_PALETTE[0],
    "log1p": SET2_PALETTE[2],
}

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11.5, 4.8),
)

for ax, validation_name in zip(
    axes,
    VALIDATION_TITLES,
):
    subset = target_plot_data[
        target_plot_data["validation"] == validation_name
    ]

    sns.barplot(
        data=subset,
        x="target",
        y="R2",
        order=target_order,
        errorbar="sd",
        palette=target_palette,
        ax=ax,
    )

    ax.axhline(
        0,
        color="black",
        linewidth=0.8,
    )

    ax.set_title(
        VALIDATION_TITLES[validation_name]
    )
    ax.set_xlabel("")
    ax.set_ylabel("R² on original target scale")

fig.suptitle(
    "Target Sensitivity — Tabular XGBoost",
    y=1.02,
)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "B_raw_vs_log1p_r2.png",
    dpi=FIGURE_DPI,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# ============================================================
# Save B target-sensitivity outputs
# ============================================================

log_fold_results.to_csv(
    OUTPUT_DIR / "log_target_fold_results_original_scale.csv",
    index=False,
)

log_oof_predictions.to_csv(
    OUTPUT_DIR / "log_target_oof_predictions_original_scale.csv",
    index=False,
)

target_comparison.to_csv(
    OUTPUT_DIR / "raw_vs_log1p_target_summary.csv",
    index=False,
)

target_delta.to_csv(
    OUTPUT_DIR / "raw_vs_log1p_target_delta.csv",
    index=False,
)

# C. PCA sensitivity / tuning inside nested CV

PCA is not fitted or selected once on the complete dataset. The table below reports the threshold selected by the **inner CV** inside each outer training fold for the ImageNet ResNet50 multimodal model. This makes the PCA decision part of the same leakage-safe model-selection procedure as the XGBoost hyperparameters.

In [ ]:
# ============================================================
# PCA structure diagnostic for raw ImageNet ResNet50 embeddings
# Descriptive only — PCA selection is performed inside nested CV
# ============================================================

X_cnn = imagenet_df[
    CNN_FEATURES
].to_numpy()

X_cnn_imputed = SimpleImputer(
    strategy="median"
).fit_transform(X_cnn)

pca_diagnostic = PCA()
pca_diagnostic.fit(
    X_cnn_imputed
)

explained_variance = (
    pca_diagnostic.explained_variance_ratio_
)

cumulative_variance = np.cumsum(
    explained_variance
)

diagnostic_thresholds = [
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95,
]

threshold_rows = []

for threshold in diagnostic_thresholds:

    n_components = (
        np.argmax(
            cumulative_variance >= threshold
        )
        + 1
    )

    threshold_rows.append({
        "variance_threshold": threshold,
        "n_components": n_components,
    })

pca_structure_table = pd.DataFrame(
    threshold_rows
)

display(
    pca_structure_table
)

pca_structure_table.to_csv(
    OUTPUT_DIR
    / "resnet50_pca_structure_diagnostic.csv",
    index=False,
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11.5, 4.8),
)

# ------------------------------------------------------------
# Scree plot
# ------------------------------------------------------------

axes[0].plot(
    np.arange(
        1,
        len(explained_variance) + 1,
    ),
    explained_variance,
)

axes[0].set_xlabel(
    "Principal component"
)

axes[0].set_ylabel(
    "Explained variance ratio"
)

axes[0].set_title(
    "ImageNet ResNet50 PCA Scree Plot"
)

# ------------------------------------------------------------
# Cumulative explained variance
# ------------------------------------------------------------

axes[1].plot(
    np.arange(
        1,
        len(cumulative_variance) + 1,
    ),
    cumulative_variance,
)

for threshold in diagnostic_thresholds:

    n_components = (
        np.argmax(
            cumulative_variance >= threshold
        )
        + 1
    )

    axes[1].axhline(
        threshold,
        linestyle="--",
        linewidth=0.8,
    )

    axes[1].scatter(
        n_components,
        threshold,
    )

    axes[1].annotate(
        f"{threshold:.0%}: "
        f"{n_components} PCs",
        (
            n_components,
            threshold,
        ),
        xytext=(5, -9),
        textcoords="offset points",
        fontsize=8,
    )

axes[1].set_xlabel(
    "Number of principal components"
)

axes[1].set_ylabel(
    "Cumulative explained variance"
)

axes[1].set_title(
    "ImageNet ResNet50 Cumulative Explained Variance"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "C_resnet50_pca_structure_diagnostic.png",
    dpi=FIGURE_DPI,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ============================================================
# Nested-CV PCA selection diagnostics from the main experiment
# ============================================================

pca_stability = (
    fusion_fold_results[
        fusion_fold_results["feature_set"]
        == "Tabular + CNN"
    ][
        [
            "validation",
            "fold",
            "R2",
            "train_R2",
            "inner_best_R2_fit_scale",
            "selected_pca_variance",
            "n_pca_components",
            "best_params",
        ]
    ]
    .sort_values(
        ["validation", "fold"]
    )
    .reset_index(drop=True)
)

pca_threshold_frequency = (
    pca_stability
    .groupby(
        [
            "validation",
            "selected_pca_variance",
        ],
        as_index=False,
    )
    .agg(
        n_outer_folds=("fold", "count"),
        mean_components=(
            "n_pca_components",
            "mean",
        ),
        mean_outer_R2=("R2", "mean"),
    )
)

pca_selection_summary = (
    pca_stability
    .groupby(
        "validation",
        as_index=False,
    )
    .agg(
        n_outer_folds=("fold", "count"),
        selected_variance_mean=(
            "selected_pca_variance",
            "mean",
        ),
        selected_variance_min=(
            "selected_pca_variance",
            "min",
        ),
        selected_variance_max=(
            "selected_pca_variance",
            "max",
        ),
        components_mean=(
            "n_pca_components",
            "mean",
        ),
        components_min=(
            "n_pca_components",
            "min",
        ),
        components_max=(
            "n_pca_components",
            "max",
        ),
    )
)

print("Fold-level nested PCA selections:")
display(pca_stability.round(4))

print("PCA threshold frequency:")
display(pca_threshold_frequency.round(4))

print("PCA selection summary:")
display(pca_selection_summary.round(4))

In [ ]:
# ============================================================
# Figure C: nested-CV PCA selections by outer fold
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11.5, 4.8),
)

for ax, validation_name in zip(
    axes,
    VALIDATION_TITLES,
):
    subset = pca_stability[
        pca_stability["validation"] == validation_name
    ].copy()

    sns.barplot(
        data=subset,
        x="fold",
        y="selected_pca_variance",
        color=SET2_PALETTE[1],
        ax=ax,
    )

    for patch, (_, row) in zip(
        ax.patches,
        subset.iterrows(),
    ):
        ax.annotate(
            f"{int(row['n_pca_components'])} PCs",
            (
                patch.get_x() + patch.get_width() / 2,
                patch.get_height(),
            ),
            ha="center",
            va="bottom",
            fontsize=8,
            xytext=(0, 3),
            textcoords="offset points",
        )

    ax.set_ylim(0.65, 0.95)
    ax.set_title(
        VALIDATION_TITLES[validation_name]
    )
    ax.set_xlabel("Outer fold")
    ax.set_ylabel("Selected explained variance")

fig.suptitle(
    "Nested-CV PCA Threshold Selection — ImageNet ResNet50",
    y=1.02,
)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "C_nested_pca_selection.png",
    dpi=FIGURE_DPI,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# ============================================================
# Save C PCA diagnostics
# ============================================================

pca_stability.to_csv(
    OUTPUT_DIR / "tuning_pca_stability.csv",
    index=False,
)

pca_threshold_frequency.to_csv(
    OUTPUT_DIR / "pca_threshold_frequency.csv",
    index=False,
)

pca_selection_summary.to_csv(
    OUTPUT_DIR / "pca_selection_summary.csv",
    index=False,
)

# D. Supplementary ResNet50 pretraining experiment

This experiment keeps the downstream architecture and validation framework fixed while changing the visual pretraining source:

- **ImageNet ResNet50** — reused directly from the main experiment;
- **Satlas ResNet50** — satellite-domain supervised pretraining;
- **SSL4EO-S12 ResNet50** — Sentinel-2 self-supervised pretraining.

The ImageNet result is not rerun. Satlas and SSL4EO-S12 are evaluated with the same outer folds, inner validation logic, XGBoost search space and nested PCA thresholds used in section A.

Because the pretrained weight families require source-appropriate image preprocessing, this is reported as a **supplementary pretraining sensitivity experiment**, not as evidence that only the weight initialization changed while every image-processing operation remained identical.

In [ ]:
# ============================================================
# Supplementary Sentinel-2 inputs and optional extraction paths
# ============================================================

BAND_PATHS = [
    RAW_DATA_DIR / "daylight21" / "processing" / "B04_k.tif",
    RAW_DATA_DIR / "daylight21" / "processing" / "B03_k.tif",
    RAW_DATA_DIR / "daylight21" / "processing" / "B02_k.tif",
]

TCI_SEARCH_ROOT = (
    RAW_DATA_DIR
    / "daylight21"
)

SATLAS_TCI_PATH = (
    TCI_SEARCH_ROOT
    / "processing"
    / "TCI_k.tif"
)

BOUNDARY_PATH = (
    RAW_DATA_DIR
    / "boundary"
    / "karachi_boundary_mask.json"
)

SUPPLEMENTARY_EMBEDDING_PATHS = {
    "ImageNet": IMAGENET_R50_PATH,
    "Satlas": SATLAS_R50_PATH,
    "SSL4EO-S12": SSL4EO_R50_PATH,
}

print("Supplementary pretraining experiment:", RUN_SUPPLEMENTARY_PRETRAINING)
print("Extract missing supplementary embeddings:", EXTRACT_MISSING_PRETRAINING_EMBEDDINGS)

for source_name, path in SUPPLEMENTARY_EMBEDDING_PATHS.items():
    print(f"{source_name}: {path}")

In [ ]:
# ============================================================
# Build or reuse the Karachi-clipped Sentinel-2 TCI mosaic
# Only used if Satlas embeddings must be extracted.
# ============================================================

def ensure_satlas_tci():
    if SATLAS_TCI_PATH.exists():
        print("Existing TCI reused:", SATLAS_TCI_PATH)
        return SATLAS_TCI_PATH

    import geopandas as gpd
    import rasterio
    from rasterio.mask import mask as rio_mask
    from rasterio.merge import merge
    from shapely.geometry import mapping

    if not BOUNDARY_PATH.exists():
        raise FileNotFoundError(
            f"Karachi boundary not found: {BOUNDARY_PATH}"
        )

    tci_10m_paths = sorted(
        TCI_SEARCH_ROOT.rglob("*TCI_10m.jp2")
    )

    if len(tci_10m_paths) == 0:
        raise FileNotFoundError(
            "No Sentinel-2 *TCI_10m.jp2 tiles were found. "
            "Satlas embeddings cannot be extracted."
        )

    print("TCI tiles found:", len(tci_10m_paths))

    boundary = gpd.read_file(
        BOUNDARY_PATH
    )

    with rasterio.open(
        tci_10m_paths[0]
    ) as src:
        raster_crs = src.crs

    boundary = boundary.to_crs(
        raster_crs
    )

    try:
        boundary_geometry = (
            boundary.geometry.union_all()
        )
    except AttributeError:
        boundary_geometry = (
            boundary.geometry.unary_union
        )

    source_files = [
        rasterio.open(path)
        for path in tci_10m_paths
    ]

    temporary_path = (
        SATLAS_TCI_PATH.parent
        / "_temporary_TCI_mosaic.tif"
    )

    try:
        mosaic, transform = merge(
            source_files,
            bounds=tuple(
                float(value)
                for value in boundary.total_bounds
            ),
        )

        profile = source_files[0].profile.copy()
        profile.update(
            driver="GTiff",
            height=mosaic.shape[1],
            width=mosaic.shape[2],
            transform=transform,
            count=3,
            dtype=mosaic.dtype,
            compress="lzw",
        )

        SATLAS_TCI_PATH.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        with rasterio.open(
            temporary_path,
            "w",
            **profile,
        ) as dst:
            dst.write(mosaic)

    finally:
        for src in source_files:
            src.close()

    with rasterio.open(
        temporary_path
    ) as src:
        clipped, clipped_transform = rio_mask(
            src,
            [mapping(boundary_geometry)],
            crop=True,
            filled=True,
            nodata=0,
        )

        clipped_profile = src.profile.copy()
        clipped_profile.update(
            height=clipped.shape[1],
            width=clipped.shape[2],
            transform=clipped_transform,
            nodata=0,
            compress="lzw",
        )

    with rasterio.open(
        SATLAS_TCI_PATH,
        "w",
        **clipped_profile,
    ) as dst:
        dst.write(clipped)

    if temporary_path.exists():
        temporary_path.unlink()

    print("TCI saved:", SATLAS_TCI_PATH)
    return SATLAS_TCI_PATH

In [ ]:
# ============================================================
# Optional Satlas / SSL4EO-S12 ResNet50 embedding extraction
# ============================================================

def extract_missing_pretraining_embedding(
    pretraining,
    output_path,
):
    import geopandas as gpd
    import rasterio
    import torch
    import torch.nn.functional as F
    from rasterio.mask import mask as rio_mask
    from torch.utils.data import DataLoader, Dataset

    try:
        from torchgeo.models import (
            ResNet50_Weights as TorchGeoResNet50_Weights,
        )
        from torchgeo.models import (
            resnet50 as torchgeo_resnet50,
        )
    except ImportError as error:
        raise ImportError(
            "The supplementary Satlas/SSL4EO experiment requires torchgeo. "
            "Install a torchgeo version that provides ResNet50_Weights."
        ) from error

    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    grid = (
        gpd.read_file(GRID_PATH)
        .sort_values("grid_id")
        .reset_index(drop=True)
    )

    grid = grid[
        grid["grid_id"].isin(BASE_FRAME["grid_id"])
    ].copy()

    grid = (
        grid
        .sort_values("grid_id")
        .reset_index(drop=True)
    )

    if len(grid) != len(BASE_FRAME):
        raise ValueError(
            "Grid geometry count does not match the final modelling sample."
        )

    class ReflectanceRGBGridDataset(Dataset):
        def __init__(self, gdf, band_paths):
            self.gdf = gdf.reset_index(drop=True)
            self.band_paths = band_paths

        def __len__(self):
            return len(self.gdf)

        def __getitem__(self, idx):
            geom = [
                self.gdf.iloc[idx].geometry
            ]

            channels = []

            for band_path in self.band_paths:
                with rasterio.open(
                    band_path
                ) as src:
                    image, _ = rio_mask(
                        src,
                        geom,
                        crop=True,
                        filled=True,
                    )

                channel = np.nan_to_num(
                    image[0],
                    nan=0.0,
                    posinf=0.0,
                    neginf=0.0,
                )
                channels.append(channel)

            image = np.stack(
                channels,
                axis=0,
            )

            image = torch.tensor(
                image,
                dtype=torch.float32,
            )

            image = F.interpolate(
                image.unsqueeze(0),
                size=(
                    SUPPLEMENTARY_PATCH_SIZE,
                    SUPPLEMENTARY_PATCH_SIZE,
                ),
                mode="bilinear",
                align_corners=False,
            ).squeeze(0)

            return (
                image,
                int(self.gdf.iloc[idx]["grid_id"]),
            )

    class TCIRGBGridDataset(Dataset):
        def __init__(self, gdf, tci_path):
            self.gdf = gdf.reset_index(drop=True)
            self.tci_path = tci_path

        def __len__(self):
            return len(self.gdf)

        def __getitem__(self, idx):
            geom = [
                self.gdf.iloc[idx].geometry
            ]

            with rasterio.open(
                self.tci_path
            ) as src:
                image, _ = rio_mask(
                    src,
                    geom,
                    crop=True,
                    filled=True,
                    indexes=[1, 2, 3],
                )

            image = np.nan_to_num(
                image,
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            )

            image = torch.tensor(
                image,
                dtype=torch.float32,
            )

            image = F.interpolate(
                image.unsqueeze(0),
                size=(
                    SUPPLEMENTARY_PATCH_SIZE,
                    SUPPLEMENTARY_PATCH_SIZE,
                ),
                mode="bilinear",
                align_corners=False,
            ).squeeze(0)

            return (
                image,
                int(self.gdf.iloc[idx]["grid_id"]),
            )

    def preprocess_batch(images, source_name):
        if source_name == "Satlas":
            return torch.clamp(
                images / 255.0,
                0.0,
                1.0,
            )

        if source_name == "SSL4EO-S12":
            images = images / 10000.0
            images = images[
                :,
                :,
                16:240,
                16:240,
            ]
            return images

        raise ValueError(
            f"Unsupported supplementary source: {source_name}"
        )

    if pretraining == "Satlas":
        tci_path = ensure_satlas_tci()
        dataset = TCIRGBGridDataset(
            grid,
            tci_path,
        )
        weights = (
            TorchGeoResNet50_Weights
            .SENTINEL2_SI_RGB_SATLAS
        )

    elif pretraining == "SSL4EO-S12":
        for path in BAND_PATHS:
            if not path.exists():
                raise FileNotFoundError(
                    f"Sentinel-2 RGB band not found: {path}"
                )

        dataset = ReflectanceRGBGridDataset(
            grid,
            BAND_PATHS,
        )
        weights = (
            TorchGeoResNet50_Weights
            .SENTINEL2_RGB_MOCO
        )

    else:
        raise ValueError(
            f"Extraction is not defined for: {pretraining}"
        )

    dataloader = DataLoader(
        dataset,
        batch_size=SUPPLEMENTARY_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    model = torchgeo_resnet50(
        weights=weights
    )

    if hasattr(model, "fc"):
        model.fc = torch.nn.Identity()
    elif hasattr(model, "reset_classifier"):
        model.reset_classifier(0)
    else:
        raise AttributeError(
            "Could not remove the ResNet50 classification head."
        )

    model = model.to(device).eval()

    for parameter in model.parameters():
        parameter.requires_grad = False

    feature_batches = []
    grid_ids = []

    print(
        f"Extracting {pretraining} ResNet50 embeddings on {device}..."
    )

    with torch.no_grad():
        for images, batch_grid_ids in dataloader:
            images = images.to(device)
            images = preprocess_batch(
                images,
                pretraining,
            )

            features = model(images)

            if features.ndim > 2:
                features = (
                    F.adaptive_avg_pool2d(
                        features,
                        output_size=1,
                    )
                    .flatten(1)
                )

            feature_batches.append(
                features.cpu().numpy()
            )
            grid_ids.extend(
                batch_grid_ids.cpu().numpy().tolist()
            )

    feature_array = np.vstack(
        feature_batches
    )

    if feature_array.shape != (
        len(grid),
        2048,
    ):
        raise ValueError(
            f"{pretraining}: expected ({len(grid)}, 2048), "
            f"found {feature_array.shape}."
        )

    feature_columns = [
        f"cnn_{index}"
        for index in range(2048)
    ]

    output_table = pd.DataFrame(
        feature_array,
        columns=feature_columns,
    )
    output_table.insert(
        0,
        "grid_id",
        grid_ids,
    )

    output_table = (
        output_table
        .sort_values("grid_id")
        .reset_index(drop=True)
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_table.to_csv(
        output_path,
        index=False,
    )

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Saved embeddings:", output_path)
    return output_table

In [ ]:
# ============================================================
# Resolve or extract supplementary pretraining embeddings
# ============================================================

supplementary_embedding_tables = {
    "ImageNet": imagenet_df.copy(),
}

if RUN_SUPPLEMENTARY_PRETRAINING:
    for pretraining, path in [
        ("Satlas", SATLAS_R50_PATH),
        ("SSL4EO-S12", SSL4EO_R50_PATH),
    ]:
        if path.exists():
            table = load_embedding_table(
                path,
                source_label=f"{pretraining} ResNet50",
                expected_dimension=2048,
            )
        elif EXTRACT_MISSING_PRETRAINING_EMBEDDINGS:
            table = extract_missing_pretraining_embedding(
                pretraining,
                path,
            )
            table = standardise_embedding_table(
                table,
                expected_dimension=2048,
                source_label=f"{pretraining} ResNet50",
            )
        else:
            raise FileNotFoundError(
                f"Missing supplementary embedding: {path}"
            )

        if set(table["grid_id"]) != set(BASE_FRAME["grid_id"]):
            raise ValueError(
                f"{pretraining} grid IDs do not match the final modelling matrix."
            )

        supplementary_embedding_tables[
            pretraining
        ] = table

print(
    "Supplementary sources ready:",
    list(supplementary_embedding_tables),
)

In [ ]:
# ============================================================
# Run D: Satlas and SSL4EO-S12 with the same nested-CV framework
# ImageNet rows are reused from section A.
# ============================================================

if RUN_SUPPLEMENTARY_PRETRAINING:
    pretraining_fold_frames = []
    pretraining_prediction_frames = []

    imagenet_main_folds = (
        fusion_fold_results[
            fusion_fold_results["feature_set"]
            == "Tabular + CNN"
        ]
        .copy()
    )
    imagenet_main_folds[
        "pretraining"
    ] = "ImageNet"

    imagenet_main_predictions = (
        fusion_predictions[
            fusion_predictions["feature_set"]
            == "Tabular + CNN"
        ]
        .copy()
    )
    imagenet_main_predictions[
        "pretraining"
    ] = "ImageNet"

    pretraining_fold_frames.append(
        imagenet_main_folds
    )
    pretraining_prediction_frames.append(
        imagenet_main_predictions
    )

    for pretraining in [
        "Satlas",
        "SSL4EO-S12",
    ]:
        embedding_table = supplementary_embedding_tables[
            pretraining
        ]

        source_df = (
            BASE_FRAME
            .merge(
                embedding_table,
                on="grid_id",
                how="inner",
                validate="one_to_one",
            )
            .sort_values("grid_id")
            .reset_index(drop=True)
        )

        for validation_name, splits in VALIDATION_SCHEMES.items():
            print("\n" + "=" * 78)
            print(
                VALIDATION_TITLES[validation_name],
                "|",
                pretraining,
                "ResNet50",
            )

            fold_result, predictions = run_nested_experiment(
                frame=source_df,
                splits=splits,
                validation_name=validation_name,
                feature_set="Tabular + CNN",
                cnn_source=f"{pretraining} ResNet50",
                cnn_features=CNN_FEATURES,
                target_mode="raw",
            )

            fold_result["pretraining"] = pretraining
            predictions["pretraining"] = pretraining

            pretraining_fold_frames.append(
                fold_result
            )
            pretraining_prediction_frames.append(
                predictions
            )

    pretraining_fold_results = pd.concat(
        pretraining_fold_frames,
        ignore_index=True,
    )

    pretraining_predictions = pd.concat(
        pretraining_prediction_frames,
        ignore_index=True,
    )

    pretraining_summary = summarise_results(
        pretraining_fold_results,
        pretraining_predictions,
        group_columns=[
            "validation",
            "pretraining",
        ],
    )

    display(
        pretraining_summary
        .sort_values(
            ["validation", "OOF_R2"],
            ascending=[True, False],
        )
        .round(4)
    )
else:
    pretraining_fold_results = pd.DataFrame()
    pretraining_predictions = pd.DataFrame()
    pretraining_summary = pd.DataFrame()
    print("Supplementary pretraining experiment skipped.")

In [ ]:
# ============================================================
# Supplementary pretraining change relative to ImageNet
# ============================================================

if not pretraining_summary.empty:
    pretraining_delta_rows = []

    for validation_name in pretraining_summary[
        "validation"
    ].unique():
        subset = pretraining_summary[
            pretraining_summary["validation"] == validation_name
        ].set_index("pretraining")

        imagenet = subset.loc["ImageNet"]

        for pretraining in [
            "Satlas",
            "SSL4EO-S12",
        ]:
            candidate = subset.loc[
                pretraining
            ]

            pretraining_delta_rows.append({
                "validation": validation_name,
                "pretraining": pretraining,
                "delta_R2_mean_vs_ImageNet": (
                    candidate["R2_mean"]
                    - imagenet["R2_mean"]
                ),
                "delta_OOF_R2_vs_ImageNet": (
                    candidate["OOF_R2"]
                    - imagenet["OOF_R2"]
                ),
                "delta_OOF_RMSE_vs_ImageNet": (
                    candidate["OOF_RMSE"]
                    - imagenet["OOF_RMSE"]
                ),
                "delta_OOF_MAE_vs_ImageNet": (
                    candidate["OOF_MAE"]
                    - imagenet["OOF_MAE"]
                ),
            })

    pretraining_delta = pd.DataFrame(
        pretraining_delta_rows
    )

    display(
        pretraining_delta.round(4)
    )
else:
    pretraining_delta = pd.DataFrame()

In [ ]:
# ============================================================
# Figure D: unified ResNet50 pretraining comparison
# ============================================================

if not pretraining_fold_results.empty:
    pretraining_order = [
        "ImageNet",
        "Satlas",
        "SSL4EO-S12",
    ]

    pretraining_palette = {
        "ImageNet": SET2_PALETTE[1],
        "Satlas": SET2_PALETTE[3],
        "SSL4EO-S12": SET2_PALETTE[4],
    }

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12.5, 4.8),
    )

    for ax, validation_name in zip(
        axes,
        VALIDATION_TITLES,
    ):
        subset = pretraining_fold_results[
            pretraining_fold_results["validation"]
            == validation_name
        ]

        sns.barplot(
            data=subset,
            x="pretraining",
            y="R2",
            order=pretraining_order,
            errorbar="sd",
            palette=pretraining_palette,
            ax=ax,
        )

        ax.axhline(
            0,
            color="black",
            linewidth=0.8,
        )

        ax.set_title(
            VALIDATION_TITLES[validation_name]
        )
        ax.set_xlabel("")
        ax.set_ylabel("R²")
        ax.tick_params(
            axis="x",
            rotation=15,
        )

    fig.suptitle(
        "Supplementary ResNet50 Pretraining Sensitivity",
        y=1.02,
    )

    plt.tight_layout()
    plt.savefig(
        SUPPLEMENTARY_FIGURE_DIR
        / "D_resnet50_pretraining_ablation_r2.png",
        dpi=FIGURE_DPI,
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
# ============================================================
# Supplementary PCA stability by pretraining source
# ============================================================

if not pretraining_fold_results.empty:
    supplementary_pca_stability = (
        pretraining_fold_results[
            [
                "validation",
                "pretraining",
                "fold",
                "R2",
                "selected_pca_variance",
                "n_pca_components",
                "inner_best_R2_fit_scale",
            ]
        ]
        .sort_values(
            [
                "validation",
                "pretraining",
                "fold",
            ]
        )
        .reset_index(drop=True)
    )

    display(
        supplementary_pca_stability.round(4)
    )
else:
    supplementary_pca_stability = pd.DataFrame()

In [ ]:
# ============================================================
# Save D supplementary pretraining outputs
# ============================================================

if not pretraining_summary.empty:
    pretraining_fold_results.to_csv(
        SUPPLEMENTARY_DIR
        / "pretraining_nested_fold_results.csv",
        index=False,
    )

    pretraining_predictions.to_csv(
        SUPPLEMENTARY_DIR
        / "pretraining_oof_predictions.csv",
        index=False,
    )

    pretraining_summary.to_csv(
        SUPPLEMENTARY_DIR
        / "pretraining_summary.csv",
        index=False,
    )

    pretraining_delta.to_csv(
        SUPPLEMENTARY_DIR
        / "pretraining_delta_vs_imagenet.csv",
        index=False,
    )

    supplementary_pca_stability.to_csv(
        SUPPLEMENTARY_DIR
        / "pretraining_pca_stability.csv",
        index=False,
    )

    print(
        "Saved supplementary pretraining outputs to:"
    )
    print(SUPPLEMENTARY_DIR)

In [ ]:
# ============================================================
# Save experiment metadata for reproducibility
# ============================================================

settings = {
    "target_column": TARGET_COLUMN,
    "main_feature_sets": [
        "Tabular",
        "Tabular + CNN",
    ],
    "main_cnn_source": "ImageNet ResNet50",
    "cnn_embedding_dimension": 2048,
    "tabular_feature_count": len(BASELINE_FEATURES),
    "tabular_features": BASELINE_FEATURES,
    "random_outer_folds": RANDOM_FOLDS,
    "spatial_outer_blocks": SPATIAL_CLUSTERS,
    "inner_folds": INNER_FOLDS,
    "pca_variance_options": PCA_VARIANCE_OPTIONS,
    "pca_selection": "inside nested CV only",
    "search_iterations": SEARCH_ITERATIONS,
    "supplementary_pretraining_sources": [
        "ImageNet",
        "Satlas",
        "SSL4EO-S12",
    ],
    "main_output_directory": str(OUTPUT_DIR),
    "imagenet_embedding_path": str(IMAGENET_R50_PATH),
    "spatial_block_path": str(SPATIAL_BLOCK_PATH),
}

with open(
    OUTPUT_DIR / "003_experiment_settings.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        settings,
        file,
        indent=2,
    )

print("Saved experiment settings.")

## Final interpretation checklist

After `Run All`:

1. **A — Final multimodal ablation:** compare `Tabular` with `Tabular + CNN` using both fold means and pooled OOF metrics. Do not claim that CNN improves the model unless the advantage is stable across the relevant metrics and validation schemes.
2. **B — Target sensitivity:** retain `log1p` only if back-transformed predictions improve on the original electricity-consumption scale. The main target remains raw unless the sensitivity results justify a change.
3. **C — PCA:** report the threshold as an inner-CV-selected hyperparameter. Do not report a single full-data PCA threshold as if it were independently validated.
4. **D — Pretraining:** treat ImageNet / Satlas / SSL4EO-S12 as a supplementary ResNet50 representation-sensitivity experiment. It should not interrupt the main Tabular vs ImageNet ResNet50 controlled comparison.
5. Notebook 05 reads the main `Tabular` OOF outputs from `work/multimodal_fusion_500m`.